# Аналитика дублей карточек ФП/СФП

Цели:
1. Оценить компании с мультидоговорностью, которая добавляет дубли строк без дедупликации.
2. Проверить повторное появление одного и того же ФП/СФП у одного ИНН в разных периодах (месяц/год).
3. Определить, создается новая карточка или обновляется существующая.
4. Посчитать, сколько дублей убирается дедупликацией (в т.ч. когда карточка одна и та же).
5. Сформировать общую и сегментную статистику.

Все секции выгружаются в отдельный Excel-файл (каждая секция на отдельном листе).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 150)

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}

# Ключ дедупликации из основной аналитики.
DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

# Период анализа можно скорректировать при необходимости.
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for root in candidates:
        if (root / "sources" / "data_crm.csv").exists():
            return root
    raise FileNotFoundError("Не найден sources/data_crm.csv")

def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)

PROJECT_ROOT = resolve_project_root()
CRM_FILE = PROJECT_ROOT / "sources" / "data_crm.csv"
OUT_XLSX = PROJECT_ROOT / "final_report" / "fp_sfp_duplicate_card_analysis.xlsx"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CRM_FILE:", CRM_FILE)
print("OUT_XLSX:", OUT_XLSX)

PROJECT_ROOT: E:\DMUKP
CRM_FILE: E:\DMUKP\sources\data_crm.csv
OUT_XLSX: E:\DMUKP\final_report\fp_sfp_duplicate_card_analysis.xlsx


In [2]:
# Загрузка и базовая подготовка данных.
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False, sep=";")
raw.columns = raw.columns.str.strip()

raw["inn"] = raw["X_INN"].apply(normalize_inn)
raw["dt"] = pd.to_datetime(raw["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
raw["segment"] = raw["X_AREA_RESP"].astype(str).str.strip().map(SEGMENT_MAP)
raw["fp_num"] = raw["NUMBER_FP_SFP"].astype(str).str.strip()
raw["fp_type"] = raw["TYPE_FP"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})
raw["agreement_num"] = raw["AGREEMENT_NUM"].astype(str).str.strip()
raw["row_id"] = raw["ROW_ID"].astype(str).str.strip()

raw = raw[(raw["dt"] >= DATE_FROM) & (raw["dt"] <= DATE_TO)].copy()
raw = raw.dropna(subset=["inn", "fp_num"]).copy()
raw = raw[raw["segment"].notna()].copy()

# Периоды для анализа повторяемости.
raw["period_month"] = raw["dt"].dt.to_period("M").astype(str)
raw["period_year"] = raw["dt"].dt.year.astype("Int64").astype(str)

print("Rows in scope:", len(raw))
print("Unique INN:", raw["inn"].nunique())

Rows in scope: 200
Unique INN: 200


## Секция 1. Компании с дублями по договорам и вклад в объем строк

Логика: внутри `DEDUP_KEY` считаем число разных договоров `agreement_num`. Если >1, считаем, что есть мультидоговорный дубль.

In [3]:
g_contract = (
    raw.groupby(DEDUP_KEY + ["segment"], as_index=False)
    .agg(
        rows_count=("inn", "size"),
        agreements_n=("agreement_num", pd.Series.nunique),
        cards_n=("row_id", pd.Series.nunique),
    )
)
g_contract["is_multicontract_dup"] = g_contract["agreements_n"] > 1
g_contract["extra_rows_if_no_dedup"] = (g_contract["rows_count"] - 1).clip(lower=0)

sec1_overall = pd.DataFrame([{
    "companies_with_multicontract_dup": raw.loc[raw.set_index(DEDUP_KEY).index.isin(g_contract.loc[g_contract["is_multicontract_dup"], DEDUP_KEY].set_index(DEDUP_KEY).index), "inn"].nunique(),
    "dedup_groups_with_multicontract_dup": int(g_contract["is_multicontract_dup"].sum()),
    "extra_rows_if_no_dedup": int(g_contract.loc[g_contract["is_multicontract_dup"], "extra_rows_if_no_dedup"].sum()),
}])

sec1_by_segment = (
    g_contract[g_contract["is_multicontract_dup"]]
    .groupby("segment", as_index=False)
    .agg(
        dedup_groups_with_multicontract_dup=("is_multicontract_dup", "size"),
        extra_rows_if_no_dedup=("extra_rows_if_no_dedup", "sum"),
    )
)

# Детализация по компаниям для секции 1.
sec1_company_detail = (
    raw.merge(g_contract[g_contract["is_multicontract_dup"]][DEDUP_KEY + ["extra_rows_if_no_dedup"]], on=DEDUP_KEY, how="inner")
    .groupby(["inn", "segment"], as_index=False)
    .agg(
        duplicate_groups=("fp_num", "size"),
        extra_rows_if_no_dedup=("extra_rows_if_no_dedup", "sum"),
    )
    .sort_values(["extra_rows_if_no_dedup", "duplicate_groups"], ascending=[False, False])
)

display(sec1_overall)
display(sec1_by_segment.head(20))
display(sec1_company_detail.head(20))

,companies_with_multicontract_dup,dedup_groups_with_multicontract_dup,extra_rows_if_no_dedup
0,0,0,0


,segment,dedup_groups_with_multicontract_dup,extra_rows_if_no_dedup


,inn,segment,duplicate_groups,extra_rows_if_no_dedup


## Секция 2. Повторяемость одного и того же ФП/СФП у ИНН по месяцам/годам

In [4]:
repeat_base = (
    raw.groupby(["inn", "segment", "fp_num", "fp_type"], as_index=False)
    .agg(
        rows_count=("inn", "size"),
        months_n=("period_month", pd.Series.nunique),
        years_n=("period_year", pd.Series.nunique),
        cards_n=("row_id", pd.Series.nunique),
    )
)
repeat_base["repeat_across_months"] = repeat_base["months_n"] > 1
repeat_base["repeat_across_years"] = repeat_base["years_n"] > 1

sec2_overall = pd.DataFrame([{
    "inn_fp_type_groups_total": len(repeat_base),
    "groups_repeat_across_months": int(repeat_base["repeat_across_months"].sum()),
    "groups_repeat_across_years": int(repeat_base["repeat_across_years"].sum()),
}])

sec2_by_segment = (
    repeat_base.groupby("segment", as_index=False)
    .agg(
        groups_total=("inn", "size"),
        repeat_across_months=("repeat_across_months", "sum"),
        repeat_across_years=("repeat_across_years", "sum"),
    )
)

sec2_detail_repeats = repeat_base[(repeat_base["repeat_across_months"]) | (repeat_base["repeat_across_years"])].copy()
sec2_detail_repeats = sec2_detail_repeats.sort_values(["months_n", "years_n", "rows_count"], ascending=[False, False, False])

display(sec2_overall)
display(sec2_by_segment)
display(sec2_detail_repeats.head(20))

,inn_fp_type_groups_total,groups_repeat_across_months,groups_repeat_across_years
0,200,0,0


,segment,groups_total,repeat_across_months,repeat_across_years
0,ККБ,55,0,0
1,МСБ,94,0,0
2,МкБ,51,0,0


,inn,segment,fp_num,fp_type,rows_count,months_n,years_n,cards_n,repeat_across_months,repeat_across_years


## Секция 3. Новая карточка или запись в существующую

Интерпретация:
- `cards_n == 1` → та же карточка (обновления/дозаписи).
- `cards_n > 1` → создавались новые карточки.

In [5]:
sec3_base = repeat_base.copy()
sec3_base["card_behavior"] = np.where(sec3_base["cards_n"] > 1, "new_cards_created", "same_card_updated")

sec3_overall = (
    sec3_base.groupby("card_behavior", as_index=False)
    .agg(groups_count=("inn", "size"), rows_count=("rows_count", "sum"), avg_cards_n=("cards_n", "mean"))
)

sec3_by_segment = (
    sec3_base.groupby(["segment", "card_behavior"], as_index=False)
    .agg(groups_count=("inn", "size"), rows_count=("rows_count", "sum"), avg_cards_n=("cards_n", "mean"))
)

sec3_detail = sec3_base.sort_values(["cards_n", "months_n", "rows_count"], ascending=[False, False, False])

display(sec3_overall)
display(sec3_by_segment.head(20))
display(sec3_detail.head(20))

,card_behavior,groups_count,rows_count,avg_cards_n
0,same_card_updated,200,200,1.0


,segment,card_behavior,groups_count,rows_count,avg_cards_n
0,ККБ,same_card_updated,55,55,1.0
1,МСБ,same_card_updated,94,94,1.0
2,МкБ,same_card_updated,51,51,1.0


,inn,segment,fp_num,fp_type,rows_count,months_n,years_n,cards_n,repeat_across_months,repeat_across_years,card_behavior
0,0067216767,ККБ,5.7.1,СФП,1,1,1,1,False,False,same_card_updated
1,0067922517,МкБ,4.2У,ФП,1,1,1,1,False,False,same_card_updated
2,0148481868,МСБ,9.12,ФП,1,1,1,1,False,False,same_card_updated
3,0154422674,МСБ,9.4.4,ФП,1,1,1,1,False,False,same_card_updated
4,0194811454,ККБ,7.1.4,СФП,1,1,1,1,False,False,same_card_updated
5,0294178396,МкБ,1.19,ФП,1,1,1,1,False,False,same_card_updated
6,0334668384,МкБ,6.8,ФП,1,1,1,1,False,False,same_card_updated
7,0344386932,МСБ,2.13.1,ФП,1,1,1,1,False,False,same_card_updated
8,035698193367,МкБ,8.13.5,СФП,1,1,1,1,False,False,same_card_updated
9,0432173245,МСБ,7.15.2,СФП,1,1,1,1,False,False,same_card_updated


## Секция 4. Сколько дублей убирает дедупликация

Считаем по ключу `DEDUP_KEY`:
- Все дубли: `rows_count - 1` в группе.
- Отдельно: дубли, где карточка одна и та же (`cards_n == 1`).

In [6]:
dedup_group = (
    raw.groupby(DEDUP_KEY + ["segment"], as_index=False)
    .agg(
        rows_count=("inn", "size"),
        cards_n=("row_id", pd.Series.nunique),
    )
)
dedup_group["duplicates_removed"] = (dedup_group["rows_count"] - 1).clip(lower=0)
dedup_group["same_card_only"] = dedup_group["cards_n"] == 1

sec4_overall = pd.DataFrame([{
    "rows_before_dedup": int(len(raw)),
    "rows_after_dedup": int(raw.drop_duplicates(DEDUP_KEY).shape[0]),
    "duplicates_removed_total": int(dedup_group["duplicates_removed"].sum()),
    "duplicates_removed_same_card_only": int(dedup_group.loc[dedup_group["same_card_only"], "duplicates_removed"].sum()),
}])

sec4_by_segment = (
    dedup_group.groupby("segment", as_index=False)
    .agg(
        dedup_groups=("segment", "size"),
        duplicates_removed_total=("duplicates_removed", "sum"),
        duplicates_removed_same_card_only=("duplicates_removed", lambda s: int(s[dedup_group.loc[s.index, "same_card_only"]].sum())),
    )
)

sec4_detail = dedup_group.sort_values(["duplicates_removed", "cards_n"], ascending=[False, False])

display(sec4_overall)
display(sec4_by_segment)
display(sec4_detail.head(20))

,rows_before_dedup,rows_after_dedup,duplicates_removed_total,duplicates_removed_same_card_only
0,200,200,0,0


,segment,dedup_groups,duplicates_removed_total,duplicates_removed_same_card_only
0,ККБ,55,0,0
1,МСБ,94,0,0
2,МкБ,51,0,0


,inn,fp_num,fp_type,IDENTIFICATION_DATE,segment,rows_count,cards_n,duplicates_removed,same_card_only
0,0067216767,5.7.1,СФП,08.01.2025,ККБ,1,1,0,True
1,0067922517,4.2У,ФП,29.07.2023,МкБ,1,1,0,True
2,0148481868,9.12,ФП,25.07.2023,МСБ,1,1,0,True
3,0154422674,9.4.4,ФП,24.12.2023,МСБ,1,1,0,True
4,0194811454,7.1.4,СФП,11.03.2025,ККБ,1,1,0,True
5,0294178396,1.19,ФП,26.06.2024,МкБ,1,1,0,True
6,0334668384,6.8,ФП,14.06.2024,МкБ,1,1,0,True
7,0344386932,2.13.1,ФП,12.05.2023,МСБ,1,1,0,True
8,035698193367,8.13.5,СФП,13.03.2023,МкБ,1,1,0,True
9,0432173245,7.15.2,СФП,02.05.2025,МСБ,1,1,0,True


## Секция 5. Общая статистика + разрез по сегментам

In [7]:
sec5_overall = pd.DataFrame([{
    "rows_total": int(len(raw)),
    "unique_inn": int(raw["inn"].nunique()),
    "unique_fp_groups_inn_fp_type": int(raw.groupby(["inn", "fp_num", "fp_type"]).ngroups),
    "unique_dedup_cards": int(raw.drop_duplicates(DEDUP_KEY).shape[0]),
}])

sec5_by_segment = (
    raw.groupby("segment", as_index=False)
    .agg(
        rows_total=("inn", "size"),
        unique_inn=("inn", pd.Series.nunique),
        unique_fp_num=("fp_num", pd.Series.nunique),
    )
)

dedup_by_segment = raw.drop_duplicates(DEDUP_KEY).groupby("segment", as_index=False).size().rename(columns={"size": "unique_dedup_cards"})
sec5_by_segment = sec5_by_segment.merge(dedup_by_segment, on="segment", how="left")

display(sec5_overall)
display(sec5_by_segment)

,rows_total,unique_inn,unique_fp_groups_inn_fp_type,unique_dedup_cards
0,200,200,200,200


,segment,rows_total,unique_inn,unique_fp_num,unique_dedup_cards
0,ККБ,55,55,50,55
1,МСБ,94,94,75,94
2,МкБ,51,51,46,51


## Экспорт в Excel (каждая секция на отдельном листе)

In [8]:
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    sec1_overall.to_excel(writer, sheet_name="S1_overall", index=False)
    sec1_by_segment.to_excel(writer, sheet_name="S1_by_segment", index=False)
    sec1_company_detail.to_excel(writer, sheet_name="S1_company_detail", index=False)

    sec2_overall.to_excel(writer, sheet_name="S2_overall", index=False)
    sec2_by_segment.to_excel(writer, sheet_name="S2_by_segment", index=False)
    sec2_detail_repeats.to_excel(writer, sheet_name="S2_repeat_detail", index=False)

    sec3_overall.to_excel(writer, sheet_name="S3_overall", index=False)
    sec3_by_segment.to_excel(writer, sheet_name="S3_by_segment", index=False)
    sec3_detail.to_excel(writer, sheet_name="S3_card_behavior", index=False)

    sec4_overall.to_excel(writer, sheet_name="S4_overall", index=False)
    sec4_by_segment.to_excel(writer, sheet_name="S4_by_segment", index=False)
    sec4_detail.to_excel(writer, sheet_name="S4_dedup_detail", index=False)

    sec5_overall.to_excel(writer, sheet_name="S5_overall", index=False)
    sec5_by_segment.to_excel(writer, sheet_name="S5_by_segment", index=False)

print(f"Сохранено: {OUT_XLSX}")

Сохранено: E:\DMUKP\final_report\fp_sfp_duplicate_card_analysis.xlsx
